# 10 · 评估体系 (维度 7.1) + RAG 评测 RAGAS (维度 4.3)

字段准确率/任务完成率/工具准确率/报价偏差 + 消融 + A/B; RAGAS: faithfulness/context_precision/context_recall/answer_relevancy。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from evaluation import metrics as M
gold = {'material':'6061','surface':'阳极氧化','quantity':50,'tolerance_grade':'IT7'}
pred = {'material':'6061','surface':'无','quantity':50,'tolerance_grade':'IT7'}
print('field_accuracy:', M.field_accuracy(pred, gold))
print('quote_deviation(115 vs 100):', M.quote_deviation(115,100))
print('tool_accuracy:', M.tool_call_accuracy(['rfq-extraction','dfm-conflict','rm-rf'],
      ['rfq-extraction','dfm-conflict','cnc-quote'], expected_seq=['rfq-extraction','dfm-conflict']))

In [ ]:
from evaluation import rag_eval as R
cases = __import__('json').loads((ROOT/'data'/'eval_set.json').read_text(encoding='utf-8'))['rag_golden']
def retrieve(q):
    for c in cases:
        if any(tok in c['query'] for tok in q.split()): return list(c['context'])
    return list(cases[0]['context'])
out = R.evaluate_set(cases, retrieve_fn=retrieve)
print('RAGAS method=', out['method'], '| n=', out['n'])
print('average=', out['average'])

## 评估运行器 (需引擎): 跑 eval_set 出聚合报告

In [ ]:
ctrl = build_controller()
ev = M.evaluate_controller(ctrl, M.load_eval_set())
print('report=', ev['report'])
print('task_completion=', ev['task_completion'])
ctrl.crm.close()